# Tick latency under WalkAround

A self-contained experiment that runs a small WalkAround workload (10 bots
for 2 minutes) against a freshly-started Minecraft server and plots the
per-tick durations emitted by the execd `minecraft_tick` collector over
time.

**Configuration**

| Component | Setting |
|-----------|---------|
| Game | `MinecraftServer` (itzg/minecraft-server:java25, Jolokia + RCON) |
| Workload | `WalkAround` |
| Bots | 10 |
| Duration | 120 s |
| Metrics path | execd `jolokia_get_minecraft_tick` -> Telegraf -> InfluxDB |

**Sweep variable:** none. This notebook captures a single configuration's
behaviour over time. For comparisons, see sibling notebooks named
`tick_latency_vs_*.ipynb`.


## Setup

Imports and experiment parameters.


In [ ]:
from datetime import timedelta
from pathlib import Path
from time import sleep

import matplotlib.pyplot as plt

import yardstick_benchmark
from yardstick_benchmark.games.minecraft.server import MinecraftServer
from yardstick_benchmark.games.minecraft.workload import WalkAround
from yardstick_benchmark.model import Node
from yardstick_benchmark.monitoring import InfluxDB, Telegraf
from yardstick_benchmark.util import wait_for_url

from _lib import query_influxdb_dataframe

NODE = Node("localhost", Path("/tmp/ysat-tick-latency"))
WORKLOAD_DURATION = timedelta(seconds=120)
BOTS = 10

# Margin after the workload's nominal end to let Telegraf flush its last
# batch (flush_interval = 10 s).
TELEGRAF_FLUSH_MARGIN_S = 15


## Run the experiment

Deploy + start the stack, run the workload, query InfluxDB for tick samples, then tear down. The DataFrame is captured before teardown so the plot cell can be re-run independently.


In [ ]:
yardstick_benchmark.clean([NODE])

influxdb = InfluxDB(NODE)
telegraf = Telegraf(NODE, jolokia=True, execd_minecraft_ticks=True)
telegraf.set_output_influxdb2(influxdb.get_info())
minecraft = MinecraftServer("yardstick-tick-latency-mc")
walkaround = WalkAround(
    NODE,
    server_host="localhost",
    duration=WORKLOAD_DURATION,
    bots_per_node=BOTS,
)

# Capture the InfluxDB DataFrame inside the try block so we have it before
# teardown stops the database. Re-running the plot cell below then doesn't
# require re-running the experiment.
ticks = None
try:
    influxdb.deploy()
    influxdb.start()
    wait_for_url(f"{influxdb.url}/health", timeout_s=60)

    # Start MC and wait for it to be fully ready (RCON listener up) before
    # starting Telegraf -- the execd minecraft_tick collector polls
    # Jolokia from its first tick, so it needs MC reachable.
    minecraft.start()
    minecraft.wait_until_ready()
    minecraft.set_world_spawn(0, 0)

    telegraf.deploy()
    telegraf.start()

    walkaround.deploy()
    walkaround.start()

    # The WalkAround entry script exits on its own after `duration`; sleep
    # for that long plus a flush margin.
    sleep(WORKLOAD_DURATION.total_seconds() + TELEGRAF_FLUSH_MARGIN_S)

    ticks = query_influxdb_dataframe(
        influxdb,
        f'''
from(bucket: "yardstick")
  |> range(start: -{int(WORKLOAD_DURATION.total_seconds()) + 60}s)
  |> filter(fn: (r) => r._measurement == "minecraft_tick" and r._field == "tick_duration_ms")
  |> keep(columns: ["_time", "_value"])
''',
    )
finally:
    walkaround.stop()
    minecraft.stop()
    telegraf.stop()
    influxdb.stop()

print(f"collected {len(ticks)} tick samples")
ticks.head()


## Plot tick duration over time


In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(ticks["_time"], ticks["_value"], linewidth=0.8)
ax.set_xlabel("time")
ax.set_ylabel("tick duration (ms)")
ax.set_title(f"Minecraft tick duration with {BOTS} WalkAround bots")
ax.grid(alpha=0.3)
ax.set_ylim(bottom=0)
fig.autofmt_xdate()
plt.show()
